# Intialize

In [ ]:
# init
import os
import time
from pathlib import Path
import boto3
import subprocess
import sys
from importlib import reload

import toolsGeneral.logger as tgl
import toolsGeneral.main as tgm
import toolsPandas.helpers as tgh
import toolsOSM.overpass as too
import toolsSync.main as tsm

def pckgs_reload():
    reload(tgm)
    reload(tgl)
    reload(too)
    reload(tgh)

pckgs_reload()

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
SAVE_DIR = DATA_DIR / 'raw/osm countries queries'
DEV_MODE = True

In [2]:
DEV_MODE

True

In [2]:
%reload_ext auto_init

# Load data to scrape

In [ ]:
# init
logger = tgl.initiate_logger('logger', DATA_DIR / 'raw/raw_scrape.log')

In [2]:
# init
osmMetaCountrDict = tgm.load(DATA_DIR / "osmMetaCountrDict.json")
print(len(osmMetaCountrDict))
process_state_file = DATA_DIR / "process_state.json"
process_state = tgm.load(process_state_file)
print(len(process_state))

241
241


In [7]:
# init
tuples = sorted(
[(k, v["id"], v["addLvlsNum"]) for (k, v) in osmMetaCountrDict.items()],
    key=lambda arg: arg[0]
)
print(len(tuples))

241


# Collect

### exclude processed countries

In [10]:
processed_countries = [country for country, country_state in process_state.items() if country_state['scrape']['status'] == 'ok']
print(len(processed_countries))
failed_countries = [country for country, country_state in process_state.items() if country_state['scrape']['status'] == 'failed']
print(len(failed_countries))

98
0


In [11]:
to_scrape_countries = [country for country, country_state in process_state.items() if country_state['scrape']['status'] in ['pending', 'error']]
to_scrape = [(country, osmMetaCountrDict[country]['id'], osmMetaCountrDict[country]['addLvlsNum']) for country in to_scrape_countries]
print(len(to_scrape))

143


### make call

In [12]:
print(f"* processed_countries: {len(processed_countries)}")
print(f"* failed_countries: {len(failed_countries)}")

* processed_countries: 98
* failed_countries: 0


In [16]:
in_chunks_countries = ['China','Armenia']

In [15]:
def scrape_country_in_chunks(tuple, save_dir, country_save_file, config, process_state, process_state_file):
    logger.info(f"* Scrape in chunks started")
    country, id, lvls = tuple
    process_state[country]['scrape']['type'] = 'chunk'
    process_state[country]['scrape']['chunk_state'] = {}
    chunk_state = process_state.get('chunk_state')

    response = too.getOSMIDAddsStruct_chunks((country, id, lvls), save_dir, chunk_state)
    logger.info(f"* Scrape in chunks finished: {response['status']} - {response['status_type']}")

    state_resume = {k:{k2:(len(v2) if type(v2) == set else v2) for k2,v2 in val.items()} for k,val in response['data'].items()}
    logger.info(f"  - Chunk status: {state_resume}")

    process_status = response['status']
    process_error = response['status_type']

    if response["status"] == "ok":
        # Try to upload data and override process status with upload result from B2
        logger.info("* Upload data to backblaze b2")
        if not DEV_MODE:
            upload_response = tsm.upload_dir_files_to_backblaze(country_save_file.parent, config)
            process_status = upload_response['status']
            process_error = upload_response['status_type']

    logger.info(f"* Update and commit to process state: {country} - {(process_status, process_error)}")
    process_state[country]['scrape']['chunk_state'] = response['data']
    tsm.update_process_state(process_state, country, 'scrape', process_status=process_status, process_error=process_error)
    tgm.dump(process_state_file, process_state)
    if not DEV_MODE:
        tsm.commit_file(process_state_file, f"Update process state: {country}: (scrape, {process_status})", config['logger'])

In [ ]:
for country, id, lvls in to_scrape:

    config = {'root':ROOT, 's3':s3, 'logger':logger}
    country_save_file = SAVE_DIR / country / f'rawOSMRes.json'
    logger.info(f"* processing: {country, id, lvls}")

    if country not in in_chunks_countries:
        
        response = too.getOSMIDAddsStruct(id, lvls)
        logger.info(f"  - Scrape for country {country} result: {response['status']}")

        # Use chunks if there are too many requests
        if '429' in response["status_type"] or 'timeout' in response["status_type"]:
            logger.info(f"  - Too many requests/timeout error, using chunks", logger)
            scrape_country_in_chunks((country, id, lvls), SAVE_DIR, country_save_file, config, process_state, process_state_file)
            continue
        
        process_status = response['status']
        process_error = response['status_type']
        # Try to upload data and override process status with upload result from B2
        if response["status"] == "ok":
            tgm.dump(country_save_file, response["data"])
            logger.info("  - Upload data to backblaze b2")
            if not DEV_MODE:
                upload_response = tsm.upload_dir_files_to_backblaze(country_save_file.parent, config)
                process_status = upload_response['status']
                process_error = upload_response['status_type']
        
        logger.info(f"  - Update and commit to process state: {country} - ({process_status, process_error})")
        tsm.update_process_state(process_state, country, 'scrape', process_status=process_status, process_error=process_error)
        tgm.dump(process_state_file, process_state)
        if not DEV_MODE:
            tsm.commit_file(process_state_file, f"Update process state: {country}: (scrape, {process_status})", config['logger'])
    else:
        scrape_country_in_chunks((country, id, lvls), SAVE_DIR, country_save_file, config, process_state, process_state_file)

    time.sleep(3)

In [ ]:
processed_countries = [country for country, country_state in process_state.items() if country_state['scrape']['status'] == 'ok']
failed_countries = [country for country, country_state in process_state.items() if country_state['scrape']['status'] == 'failed']

print(f"* new total of processed_countries: {len(processed_countries)}")
print(f"* new total of failed_countries: {len(failed_countries)}")

# Review

In [2]:
country_dirs = [f for f in (DATA_DIR / 'raw/osm countries queries').glob('*') if f.is_dir()]
print(len(country_dirs))

97


In [3]:
raw_by_cntr = {}
# for chunks and non chunk files
for dir in country_dirs:
    files_elements = [tgm.load(str(f))['elements'] for f in dir.glob('*.json')]
    elements = [ele for list in files_elements for ele in list]
    raw_by_cntr[str(dir.name)] = elements

print(f"* Number of countries with raw data: {len(raw_by_cntr)}")

* Number of countries with raw data: 97


In [17]:
resume = {}
for country, raw in raw_by_cntr.items():
    resume[country] = {}
    resume[country]['lvl_1'] = len([ele for ele in raw if ele['tags']['admin_level'] == '4'])
    resume[country]['lvl_2'] = len([ele for ele in raw if ele['tags']['admin_level'] == '6'])
    resume[country]['lvl_3'] = len([ele for ele in raw if ele['tags']['admin_level'] == '8'])
    resume[country]['total'] = len([ele for ele in raw if ele['tags']['admin_level'] != '2'])

resume = pd.DataFrame(resume).T
resume.peek()

,lvl_1,lvl_2,lvl_3,total
Afghanistan,34,0,0,34
AkrotiriAndDhekelia,0,0,0,0
Albania,3,12,374,389
Algeria,58,547,1542,2147
Andorra,0,0,0,0
Angola,19,29,15,63
Anguilla,0,0,0,0
AntiguaAndBarbuda,3,8,0,11
Argentina,25,661,524,1210
Australia,15,600,0,615
